# Recursive Contracts and Divide-and-Conquer


```{contents}
:local:
:depth: 2
```


```{index} Let a Helper Promise Exactly One Smaller Problem
```



```{index} Let a Helper Promise Exactly One Smaller Problem
```

## Let a Helper Promise Exactly One Smaller Problem



A public method can accept a complete report while a private recursive helper accepts a suffix or range. The helper contract must name that smaller problem precisely. For a suffix starting at index i, the result might be the total of entries i through the end. The public caller starts with i zero; other starting indexes are an implementation detail rather than a burden placed on every client.

The base case is an empty suffix, not a special final value guessed from one example. Its sum is zero. For a nonempty suffix, combine the first available entry with the helper's result on the remaining suffix. The progress measure n-i decreases by one, which connects the contract to termination. The caller still needs an arithmetic-range policy: termination does not prove that a fixed-width sum cannot overflow.

For divide-and-conquer, represent a range as [lo, hi). Its length is hi-lo, and empty means lo equals hi. Splitting at lo+(hi-lo)/2 gives two adjacent half-open ranges. On five items, the lengths are two and three; neither item duplication nor equal lengths are required. What matters is complete, nonoverlapping coverage and strictly smaller children when recursion continues.

| Design question | What to state before recursing |
| --- | --- |
| What does the helper return? | A result for this suffix/range |
| When is the answer direct? | Empty/singleton or another explicit base |
| What becomes smaller? | Remaining length or another well-founded measure |
| How are child results combined? | An operation justified by the parent contract |

Guided contrast: finding a maximum needs results from both halves, while a binary-search comparison chooses only one remaining half. Both use interval splitting, but they do not have the same call tree or work recurrence. Predict child calls for one size-five input under each algorithm before comparing their costs.

For related explanation, see the further reading below.[^development-reading]


```{index} Recursive Problem Solving
```

<span id="recursive-problem-solving"></span>


Basic recursive cases, base answers, and call/return tracing are taught in {doc}`Chapter 17 <../17_trees/1701_tree_structure>`. Review that section before this optional extension. Here you develop helper contracts, progress arguments, resource limits, and divide-and-conquer design.


```{index} Progress Measures
```

## Progress Measures




```{index} Progress Measures
```

A progress measure proves that recursion is moving in the right direction. The measure does not have to be a single number, but it must get closer to a base case.

In [ ]:
using System;
using System.Linq;

int[] values={2,4,6,8,10};
Console.WriteLine(Contains(values,6));Console.WriteLine(Contains(values,7));
Console.WriteLine(Contains(Array.Empty<int>(),6));
bool Contains(int[] values,int target)
{
 ArgumentNullException.ThrowIfNull(values);
 return Scan(0);
 bool Scan(int index)
 {
  if(index==values.Length)return false;
  if(values[index]==target)return true;
  return Scan(index+1);
 }
}

True
False
False


The wrapper hides the helper index so clients cannot supply a negative or out-of-range start. Scan(index) promises whether the target occurs in the suffix [index,n). An empty suffix returns false; a matching first item returns true; otherwise the answer comes from the smaller suffix. The measure n−index decreases on every recursive call. Empty input needs no array access. Worst-case time and stack are Θ(n); a first-item match needs only one helper frame.

**Common mistake.** Letting the helper take the whole array as its parameter and recursing on a copy without the first element. Each call then allocates a new array, so the search costs Θ(n²) copying for a Θ(n) scan; an index into the shared array is the progress measure that costs nothing to advance.


```{index} Recursion and Iteration
```

## Recursion and Iteration




```{index} Recursion and Iteration
```

Many recursive methods can be written with loops. Use recursion when the problem naturally breaks into smaller problems of the same shape, or when recursion makes the structure easier to see.

In [ ]:
using System;
using System.Linq;

int[] values={10,-3,8};
Console.WriteLine($"recursive={Sum(values)}, iterative={values.Sum(x=>(long)x)}");
long Sum(int[] values)
{
 ArgumentNullException.ThrowIfNull(values);
 return From(0);
 long From(int index)=>index==values.Length?0:values[index]+From(index+1);
}

recursive=15, iterative=15


Both sums visit every item, taking Θ(n) time under the integer cost model. The recursive version retains Θ(n) active frames; an iterative sum retains Θ(1) auxiliary state. Recursion changes expression of control flow, not the amount of data that must be processed. Prefer the loop for long flat reports; recursion is particularly useful for balanced splits and nested structures.

**Common mistake.** Using recursion for a long flat report because it "reads better". Each item adds a stack frame, so a list of a few hundred thousand amounts can exhaust the runtime stack where the loop needs one variable; recursion earns its cost on nested or halving structures, not on flat sequences.


```{index} Recursive Risks
```

## Recursive Risks




```{index} Recursive Risks
```

A recursive method with a missing base case or no progress can call itself forever until the program runs out of stack space. In C#, recursive calls are not something to use casually for huge linear loops.

In [ ]:
using System;

Console.WriteLine(Countdown(3));
foreach(int n in new[]{-1,101})
{
 try{Countdown(n);}catch(ArgumentOutOfRangeException){Console.WriteLine($"reject {n}");}
}
int Countdown(int n)
{
 if(n<0||n>100)throw new ArgumentOutOfRangeException(nameof(n),"Small teaching input required.");
 return n==0?0:1+Countdown(n-1);
}


3
reject -1
reject 101


Check input domain, direct base answer, and a well-founded progress measure before running recursive code. Do not intentionally trigger stack exhaustion in the browser. Tail-call elimination is not guaranteed for these C# methods. A finite domain and arithmetic range are separate from the proof of termination; validate both when designing the interface.

**Common mistake.** Relying on the exception to stop a runaway recursion. A `StackOverflowException` cannot be caught and ends the process; the domain check in `Countdown` runs *before* the first recursive call, which is the only place a guard can protect the stack.


```{index} Exercise
```

## Exercise




```{index} Exercise
```

Recursively count positive expense adjustments in a suffix. Zero is not positive. State the helper postcondition and progress measure; test empty input.

In [ ]:
using System;
using System.Linq;

int[] values={-2,0,5,7};
// Your code starts here.
/* TODO */
// Your code ends here.

In [1]:
using System;
using System.Linq;

// Solution
int[] values={-2,0,5,7};
Console.WriteLine($"positive={Count(values)}, empty={Count(Array.Empty<int>())}");
int Count(int[] values)
{
 return From(0);
 // From(i) counts positive values in [i,n); measure n-i.
 int From(int i)=>i==values.Length?0:(values[i]>0?1:0)+From(i+1);
}

positive=2, empty=0


The output is `positive=2, empty=0`. The 0 was not counted, because the test is `> 0`, and the empty array returned 0 from the base case without reading any element.


```{index} Divide-and-Conquer Design
```

<span id="divide-and-conquer-design"></span>


Divide-and-conquer algorithms solve a problem in three phases: divide the input into smaller parts, solve the smaller parts, and combine their results. Recursion is a natural implementation technique because each subproblem has the same shape as the original problem.


```{index} The Pattern
```

## The Pattern




```{index} The Pattern
```

A divide-and-conquer method usually follows this shape:

1. If the problem is small enough, solve it directly.
2. Divide the input into smaller pieces.
3. Recursively solve the pieces.
4. Combine the partial answers.

In [ ]:
using System;
using System.Linq;

foreach(int[] values in new[]{new[]{3,9,2,7,5},new[]{-8},Array.Empty<int>()})
{
 try{Console.WriteLine($"maximum={Max(values)}");}
 catch(ArgumentException){Console.WriteLine("empty maximum rejected");}
}
int Max(int[] values)
{
 ArgumentNullException.ThrowIfNull(values);
 if(values.Length==0)throw new ArgumentException("Nonempty input required.");
 return Solve(0,values.Length);
 int Solve(int start,int end)
 {
  if(end-start==1)return values[start];
  int middle=start+(end-start)/2;
  return Math.Max(Solve(start,middle),Solve(middle,end));
 }
}

maximum=9
maximum=-8
empty maximum rejected


The helper promise is the maximum of the nonempty half-open range [start,end). The wrapper rejects empty input before recursion. For length>1, both child ranges are nonempty, smaller, disjoint, and together cover the parent, including odd lengths. Inductively correct child maxima combine to the parent maximum. The size measure end−start decreases. T(n)=T(floor(n/2))+T(ceil(n/2))+Θ(1), hence Θ(n) work and Θ(log n) stack. A loop also takes Θ(n) work with less stack: splitting alone does not make maximum selection faster.

**Common mistake.** Splitting as `Solve(start, middle)` and `Solve(middle + 1, end)`. The element at `middle` belongs to neither half, so one value is silently skipped; with half-open ranges the second call starts exactly at `middle`.


```{index} Halving Strategies
```

## Halving Strategies




```{index} Halving Strategies
```

Some algorithms solve only one half after dividing. Binary search is the classic example: compare with the middle value, then search the half where the target could still be.

In [ ]:
using System;
using System.Linq;

int[] sorted={2,4,6,8,10,12,14};
foreach(int target in new[]{2,10,14,5})Console.WriteLine($"target={target}, index={Search(sorted,target)}");
Console.WriteLine($"empty={Search(Array.Empty<int>(),5)}");
int Search(int[] values,int target)
{
 ArgumentNullException.ThrowIfNull(values);
 return Find(0,values.Length);
 int Find(int start,int end)
 {
  if(start==end)return -1;
  int middle=start+(end-start)/2;
  if(values[middle]==target)return middle;
  return target<values[middle]?Find(start,middle):Find(middle+1,end);
 }
}

target=2, index=0
target=10, index=4
target=14, index=6
target=5, index=-1
empty=-1


Precondition: input is sorted in ascending order and is not modified during the search. The result is any matching index, or −1 when absent; duplicates do not promise a first match. The candidate interval is [start,end). Sortedness justifies discarding the impossible side. Excluding middle from the right interval prevents repeating a one-item range. Each surviving range is smaller; an empty range stops recursion. Only one child is explored, giving T(n)=T(floor(n/2))+Θ(1) as a size model, Θ(log n) worst-case time/stack, and Θ(1) best-case time. Checking sortedness on every call would add linear work.

**Common mistake.** Recursing on `Find(middle, end)` after learning that the target is larger than `values[middle]`. A one-element range then calls itself with the same bounds forever; excluding the middle on the right side is what guarantees the range shrinks.


```{rubric} Footnotes
```


[^2201_recursion-1]: Tail-call optimization is not something introductory C# programmers should rely on. Prefer clear loops for simple linear repetition when stack depth could become large.
[^2201_recursion-2]: A recursive method can have more than one base case. The important point is that every path of recursive calls eventually reaches one of them.


[^development-reading]: Further reading: [Nakov and colleagues, Methods](https://introprogramming.info/english-intro-csharp-book/read-online/chapter-9-methods/). The worked business scenarios and guided reasoning here are original teaching examples.
